# Colab Baseline cot_structured 32768 vLLM Eval

Run the base `Qwen/Qwen3-4B-Thinking-2507` model with `cot_structured_only`, `vllm`, `bf16`, and `32768` generation tokens. This notebook intentionally does not load a LoRA/SFT adapter.

In [ ]:
!nvidia-smi

## Google Drive

In [ ]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ModuleNotFoundError:
    IN_COLAB = False
    print('Not running in Colab; skipping Google Drive mount.')

## Repo Setup

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = 'https://github.com/DarinAnthony/151B_SP26_Competition.git'
cwd = Path.cwd().resolve()

if IN_COLAB:
    REPO_DIR = Path('/content/151B_SP26_Competition')
    if not REPO_DIR.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
elif (cwd / 'pyproject.toml').exists():
    REPO_DIR = cwd
elif (cwd.parent / 'pyproject.toml').exists():
    REPO_DIR = cwd.parent
else:
    raise RuntimeError(f'Could not find repo root from {cwd}')

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('Repo:', REPO_DIR)
print('Python:', sys.executable)

## Dependencies

Run this once per fresh Colab runtime. Restart the runtime if Colab asks after installing `vllm`, then rerun the setup cells.

In [ ]:
%pip install -q "hydra-core>=1.3" vllm datasets accelerate transformers tqdm sympy pyyaml wandb

## Baseline Run Config

No adapter is configured in this notebook. Results are written directly to Google Drive when running in Colab.

In [ ]:
from pathlib import Path
import os

RESULTS_DIR = Path(os.environ.get(
    'RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else '/cephfs/qwen_math_comp/eval_results',
))
DRIVE_RESULTS_DIR = Path(os.environ.get(
    'DRIVE_RESULTS_DIR',
    '/content/drive/MyDrive/qwen_math_comp/eval_results' if IN_COLAB else str(RESULTS_DIR),
))
RUN_NAME = os.environ.get('RUN_NAME', 'baseline_cot_structured_32768_vllm_4090')
WANDB_NAME = os.environ.get('WANDB_NAME', RUN_NAME)
MAX_TOKENS = int(os.environ.get('MAX_TOKENS', '32768'))
RUNNER_MICRO_BATCH_SIZE = os.environ.get('RUNNER_MICRO_BATCH_SIZE', '1')
RUNNER_PARALLEL_SAMPLES = os.environ.get('RUNNER_PARALLEL_SAMPLES', '1')

os.environ['WANDB_NAME'] = WANDB_NAME
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE
os.environ['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES

print('RESULTS_DIR =', RESULTS_DIR)
print('DRIVE_RESULTS_DIR =', DRIVE_RESULTS_DIR)
print('RUN_NAME =', RUN_NAME)
print('WANDB_NAME =', os.environ['WANDB_NAME'])
print('MAX_TOKENS =', MAX_TOKENS)
print('RUNNER_MICRO_BATCH_SIZE =', os.environ['RUNNER_MICRO_BATCH_SIZE'])
print('RUNNER_PARALLEL_SAMPLES =', os.environ['RUNNER_PARALLEL_SAMPLES'])
print('WANDB_PROJECT =', os.environ.get('WANDB_PROJECT', '<unset; W&B disabled>'))

## Eval Helper

Equivalent full-eval shell command:

```bash
python -m experiments.prompt_engineering.src.eval \
  run=cot_structured_only \
  eval.max_tokens=32768 \
  runner.engine=vllm \
  runner.quant=bf16 \
  results_dir=/content/drive/MyDrive/qwen_math_comp/eval_results \
  run_name=baseline_cot_structured_32768_vllm_4090
```

In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import sys
import time

def build_eval_cmd(run_name, *, max_tokens, results_dir, slice_indices=None):
    cmd = [
        sys.executable,
        '-m', 'experiments.prompt_engineering.src.eval',
        'run=cot_structured_only',
        f'eval.max_tokens={max_tokens}',
        'runner.engine=vllm',
        'runner.quant=bf16',
        f'results_dir={results_dir}',
        f'run_name={run_name}',
    ]
    if slice_indices is not None:
        joined = ','.join(str(int(i)) for i in slice_indices)
        cmd.insert(4, f'eval.slice_indices=[{joined}]')
    return cmd

def backup_run_to_drive(run_name, *, results_dir=RESULTS_DIR, drive_results_dir=DRIVE_RESULTS_DIR):
    src = Path(results_dir) / run_name
    dst = Path(drive_results_dir) / run_name
    if not src.exists():
        print(f'No run directory to back up yet: {src}')
        return
    if src.resolve() == dst.resolve():
        print(f'Results are already on Drive: {src}')
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'Backed up results to Google Drive: {dst}')

def run_eval(run_name, *, max_tokens=MAX_TOKENS, results_dir=RESULTS_DIR, slice_indices=None):
    env = os.environ.copy()
    env['WANDB_NAME'] = WANDB_NAME
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['EVAL_METRICS_DIR'] = str(DRIVE_RESULTS_DIR)
    env['RUNNER_PARALLEL_SAMPLES'] = RUNNER_PARALLEL_SAMPLES
    env['RUNNER_MICRO_BATCH_SIZE'] = RUNNER_MICRO_BATCH_SIZE

    cmd = build_eval_cmd(
        run_name,
        max_tokens=max_tokens,
        results_dir=results_dir,
        slice_indices=slice_indices,
    )

    print('WANDB_NAME =', env['WANDB_NAME'])
    print('DRIVE_RESULTS_DIR =', env['EVAL_METRICS_DIR'])
    print('RUNNER_MICRO_BATCH_SIZE =', env['RUNNER_MICRO_BATCH_SIZE'])
    print('RUNNER_PARALLEL_SAMPLES =', env['RUNNER_PARALLEL_SAMPLES'])
    print('Command:', ' '.join(shlex.quote(part) for part in cmd))

    start = time.time()
    subprocess.run(cmd, cwd=REPO_DIR, env=env, check=True)
    print(f'Elapsed: {(time.time() - start) / 60:.1f} min')

## 5-Item Smoke Test

In [ ]:
run_eval(
    f'{RUN_NAME}_smoke5',
    max_tokens=MAX_TOKENS,
    slice_indices=[0, 1, 2, 3, 4],
)

## Full Standard Eval

In [ ]:
run_eval(
    RUN_NAME,
    max_tokens=MAX_TOKENS,
)

## Result Files

In [ ]:
results_root = Path(RESULTS_DIR)
if not results_root.exists():
    print(f'No results directory yet: {results_root}')
else:
    for path in sorted(results_root.glob('*'))[-10:]:
        print(path)
        for metrics_name in ['metrics.json', 'leaderboard.csv', 'leaderboard.jsonl']:
            metrics_path = path / metrics_name
            if metrics_path.exists():
                print('  ', metrics_path)

## Back Up Results to Google Drive

In [ ]:
# Run this after smoke or full eval. In Colab, RESULTS_DIR is already on Drive.
backup_run_to_drive(f'{RUN_NAME}_smoke5')
backup_run_to_drive(RUN_NAME)
